# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

In [3]:
df = pd.read_csv(f"../../data/top30groups/OneHotLongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Weapon type prediction

In [6]:
torch.cuda.empty_cache()


In [7]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
import random

param_grid = {
    'lr': [0.01, 0.001],
    'n_tree': [10, 20, 50, 80, 100],
    'tree_depth': [8, 9, 10, 11, 12, 13, 14],
    'tree_feature_rate': [0.1, 0.3, 0.5],
    'feat_dropout': [0.0, 0.2, 0.1],
    'out_size_nrf': [64, 128, 256, 512]
    }

# Convert to list of dicts (cartesian product)
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
sampled_combos = random.sample(grid_combos, 20)

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")
    best_run = None
    best_score = -1

    for combo in sampled_combos:
        args = {
            **dict(zip(param_names, combo)),
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 1500,
            'final_evaluation': False
        }

        print(f"Running config: {args}")
        acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs =train_joint(df, args, label_index, verbose=True)

        if acc > best_score:
            best_score = acc
            best_run = {
                "args": args,
                "acc": acc,
                "epoch": epoch,
                "y_pred": y_pred_decoded,
                "y_true": y_true_decoded,
                "precision": p,
                "recall": r,
                "f1": f1,
                "micro": (p_micro, r_micro, f1_micro),
                "macro": (p_macro, r_macro, f1_macro),
                "auroc": (auc_w, auc_mi, auc_ma),
                "epoch_logs": epoch_logs
            }


    # Save best results
    if best_run:
        args = best_run["args"]
        os.makedirs(f"Results{partition}", exist_ok=True)

        final_args = {**args, "final_evaluation": True, "epochs": 3000}
        print(f"\nRerunning best config with final evaluation: {final_args}")
        acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(df, final_args, label_index, verbose=True)

        results_path = f"Results{partition}/Results_{col}_prediction"
        with open(results_path, "w") as f:
            f.write(f"Best acc: {acc:.4f} at epoch {epoch} for {col} prediction\n")
            f.write(f"Config: {final_args}\n")
            f.write(f"Weighted Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}\n")
            f.write(f"Macro Precision: {p_macro:.4f}, Recall: {r_macro:.4f}, F1: {f1_macro:.4f}\n")
            f.write(f"Micro Precision: {p_micro:.4f}, Recall: {r_micro:.4f}, F1: {f1_micro:.4f}\n")
            f.write(f"AUROC Weighted: {auc_w:.4f}, Micro: {auc_mi:.4f}, Macro: {auc_ma:.4f}\n")

        log_path = f"Results{partition}/epoch_logs_{col}_prediction"
        with open(log_path, "w") as f:
            f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

        y_preds.append(best_run['y_pred'])
        y_trues.append(best_run['y_true'])

print(best_score)


Training model for weaptype1 prediction...
Running config: {'lr': 0.001, 'n_tree': 100, 'tree_depth': 9, 'tree_feature_rate': 0.5, 'feat_dropout': 0.2, 'out_size_nrf': 512, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Epoch 000 | NRF Loss: 3.4012 | Val Acc: 0.0683
Epoch 050 | NRF Loss: 2.8880 | Val Acc: 0.4567
Epoch 100 | NRF Loss: 2.7028 | Val Acc: 0.4483
Epoch 150 | NRF Loss: 2.5742 | Val Acc: 0.4467
Epoch 200 | NRF Loss: 2.4609 | Val Acc: 0.4467
Epoch 250 | NRF Loss: 2.3561 | Val Acc: 0.4450
Epoch 300 | NRF Loss: 2.2568 | Val Acc: 0.4383
Epoch 350 | NRF Loss: 2.1616 | Val Acc: 0.4367
Epoch 400 | NRF Loss: 2.0697 | Val Acc: 0.4383
Epoch 450 | NRF Loss: 1.9815 | Val Acc: 0.4383
Epoch 500 | NRF Loss: 1.8966 | Val Acc: 0.4367
Epoch 550 | NRF Loss: 1.8148 | Val Acc: 0.4367
Epoch 600 | NRF Loss: 1.7360 | Val Acc: 0.4333
Epoch 650 | NRF Loss: 1.6599 | Val Acc: 0.4333
Epoch 700 | NRF Loss: 1.5864 | Val Acc: 0.4350
Epoch 750 | NRF Loss: 1.5157 | Val Acc: 

KeyboardInterrupt: 

In [ ]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [ ]:
print(best_run)

{'args': {'lr': 0.01, 'n_tree': 80, 'tree_depth': 14, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'out_size_nrf': 2048, 'partition': 'gtd100', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}, 'acc': 0.4716666638851166, 'epoch': 756, 'y_pred': [], 'y_true': [], 'precision': 0.6971783638000488, 'recall': 0.4716666638851166, 'f1': 0.5178345441818237, 'micro': (0.4716666638851166, 0.4716666638851166, 0.4716666638851166), 'macro': (0.6971783638000488, 0.47166669368743896, 0.5178345441818237), 'auroc': (0.8744109195402299, 0.8797675766283525, 0.8744109195402298), 'epoch_logs': [1.2403643131256104, 1.2363746166229248, 1.2500090599060059, 1.3692607879638672, 1.2004399299621582, 1.2128515243530273, 1.2070555686950684, 1.1987426280975342, 1.3353848457336426, 1.2036876678466797, 1.1980769634246826, 1.1866919994354248, 1.2017719745635986, 1.38417649269104, 1.2216806411743164, 1.209977388381958, 1.2147307395935059, 1.1881699562072754, 1.208287239074707, 1.4517924785614014, 1.3364806175

In [ ]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [ ]:
best_acc

NameError: name 'best_acc' is not defined

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])